In [1]:
# ============================================================
# SMART NEXT-WORD PREDICTOR
# Using WikiText-2 Dataset from Kaggle
# ============================================================

import pandas as pd
import re
from collections import Counter


# ============================================================
# STEP 1: LOAD DATASET
# ============================================================

# Keep train.txt in the same folder as this notebook
file_path = "train.txt"

with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()

print("WikiText-2 dataset loaded successfully!")
print("Total characters:", len(text))


# ============================================================
# STEP 2: CLEAN THE TEXT
# ============================================================

# Convert text to lowercase
text = text.lower()

# Remove unwanted characters
text = re.sub(r"[^a-zA-Z\s]", " ", text)

# Remove extra spaces
text = re.sub(r"\s+", " ", text).strip()

print("\nText cleaning completed!")


# ============================================================
# STEP 3: TOKENIZATION
# ============================================================

tokens = text.split()

print("Total tokens:", len(tokens))

print("\nFirst 20 tokens:")
print(tokens[:20])


# ============================================================
# STEP 4: BUILD UNIGRAM FREQUENCY TABLE
# ============================================================

unigram_counts = Counter(tokens)

print("\nTop 10 Unigrams:")
print(unigram_counts.most_common(10))


# ============================================================
# STEP 5: BUILD BIGRAM FREQUENCY TABLE
# ============================================================

bigram_counts = Counter(
    zip(tokens[:-1], tokens[1:])
)

print("\nTop 10 Bigrams:")
print(bigram_counts.most_common(10))


# ============================================================
# STEP 6: BUILD TRIGRAM FREQUENCY TABLE
# ============================================================

trigram_counts = Counter(
    zip(tokens[:-2], tokens[1:-1], tokens[2:])
)

print("\nTop 10 Trigrams:")
print(trigram_counts.most_common(10))


# ============================================================
# STEP 7: CALCULATE PROBABILITIES
# ============================================================

total_words = len(tokens)


# Unigram probability
def unigram_probability(word):

    return unigram_counts[word] / total_words


# Bigram probability
def bigram_probability(previous_word, word):

    previous_count = unigram_counts[previous_word]

    if previous_count == 0:
        return 0

    return (
        bigram_counts[(previous_word, word)]
        / previous_count
    )


# Trigram probability
def trigram_probability(word1, word2, word3):

    previous_count = bigram_counts[(word1, word2)]

    if previous_count == 0:
        return 0

    return (
        trigram_counts[(word1, word2, word3)]
        / previous_count
    )


# ============================================================
# STEP 8: NEXT-WORD PREDICTION FUNCTION
# ============================================================

def predict_next_words(sentence, top_n=5):

    # Convert sentence to lowercase
    sentence = sentence.lower()

    # Remove unwanted characters
    sentence = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        sentence
    )

    # Remove extra spaces
    sentence = re.sub(
        r"\s+",
        " ",
        sentence
    ).strip()

    # Tokenize
    input_words = sentence.split()

    if len(input_words) == 0:
        return []


    candidates = []


    # --------------------------------------------------------
    # TRIGRAM PREDICTION
    # --------------------------------------------------------

    if len(input_words) >= 2:

        word1 = input_words[-2]
        word2 = input_words[-1]

        for (w1, w2, next_word), count in trigram_counts.items():

            if w1 == word1 and w2 == word2:

                probability = trigram_probability(
                    word1,
                    word2,
                    next_word
                )

                candidates.append(
                    (next_word, probability)
                )


    # --------------------------------------------------------
    # BIGRAM PREDICTION
    # --------------------------------------------------------

    if len(candidates) == 0:

        last_word = input_words[-1]

        for (previous_word, next_word), count in bigram_counts.items():

            if previous_word == last_word:

                probability = bigram_probability(
                    previous_word,
                    next_word
                )

                candidates.append(
                    (next_word, probability)
                )


    # --------------------------------------------------------
    # UNIGRAM FALLBACK
    # --------------------------------------------------------

    if len(candidates) == 0:

        candidates = [
            (
                word,
                unigram_probability(word)
            )
            for word in unigram_counts
        ]


    # Sort candidates by probability
    candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )


    # Remove duplicate words
    result = []
    seen = set()

    for word, probability in candidates:

        if word not in seen:

            result.append(
                (word, probability)
            )

            seen.add(word)

        if len(result) == top_n:
            break

    return result


# ============================================================
# STEP 9: USER INPUT
# ============================================================

print("\n")
print("=" * 60)
print("SMART NEXT-WORD PREDICTOR")
print("=" * 60)

query = input(
    "\nEnter a sentence or partial sentence: "
)


# ============================================================
# STEP 10: PREDICT NEXT WORDS
# ============================================================

predictions = predict_next_words(
    query,
    top_n=5
)


# ============================================================
# STEP 11: DISPLAY RESULTS
# ============================================================

print("\nInput Sentence:")
print(query)

print("\nTop 5 Next-Word Predictions:")

if len(predictions) == 0:

    print("No prediction found.")

else:

    for i, (word, probability) in enumerate(
        predictions,
        start=1
    ):

        print(
            f"{i}. {word} "
            f"(Probability: {probability:.4f})"
        )


# ============================================================
# STEP 12: COMPLETION MESSAGE
# ============================================================

print("\n")
print("=" * 60)
print("NEXT-WORD PREDICTION COMPLETED SUCCESSFULLY!")
print("=" * 60)

HTTPError: HTTP Error 301: Moved Permanently